# OmniSub 2026 — Precomputed Results Submission

Lightweight CPU notebook: loads precomputed `results.json` from dataset and writes `submission.csv`.
No GPU, no internet required.

In [ ]:
# ── Cell 1: Load precomputed results ──
import os, json, glob

# Find results.json in attached datasets
results_path = None
for pattern in [
    '/kaggle/input/**/results.json',
    '/kaggle/input/omnisub-precomputed-results/results.json',
    '/kaggle/input/omnisub-precomputed-results/*.json',
]:
    matches = glob.glob(pattern, recursive=True)
    for m in matches:
        if 'results' in os.path.basename(m).lower():
            results_path = m
            break
    if results_path:
        break

if not results_path:
    # Fallback: any JSON in the dataset
    all_json = glob.glob('/kaggle/input/**/*.json', recursive=True)
    for j in all_json:
        if 'kernel-metadata' not in j and 'dataset-metadata' not in j:
            results_path = j
            break

assert results_path, 'results.json not found in /kaggle/input/'
print(f'Loading: {results_path}')

with open(results_path) as f:
    results = json.load(f)

print(f'Loaded {len(results)} transcriptions')
# Show sample
for i, (path, text) in enumerate(results.items()):
    if i >= 5: break
    print(f'  {path} → "{text[:60]}"')

In [ ]:
# ── Cell 2: Write submission.csv ──
import csv, re, glob

# Find sample_submission.csv for correct path ordering
sample_sub = None
for pattern in [
    '/kaggle/input/omni-sub/sample_submission.csv',
    '/kaggle/input/**/sample_submission.csv',
]:
    matches = glob.glob(pattern, recursive=True)
    if matches:
        sample_sub = matches[0]
        break
assert sample_sub, 'sample_submission.csv not found'
print(f'Sample submission: {sample_sub}')

# Read test paths in order
test_paths = []
with open(sample_sub) as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    for row in reader:
        test_paths.append(row[0])
print(f'Test paths: {len(test_paths)}')

def norm(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Write submission
OUTPUT = '/kaggle/working/submission.csv'
missing = 0
with open(OUTPUT, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['path', 'transcription'])
    for path in test_paths:
        text = results.get(path, '')
        if not text:
            text = 'a'
            missing += 1
        writer.writerow([path, norm(text)])

if missing:
    print(f'WARNING: {missing} paths had no precomputed result — filled with "a"')

# Verify
import pandas as pd
sub = pd.read_csv(OUTPUT)
print(f'Shape: {sub.shape}')
print(f'Nulls: {sub["transcription"].isna().sum()}')
print(f'Empty: {(sub["transcription"] == "").sum()}')
sub['wc'] = sub['transcription'].apply(lambda x: len(str(x).split()))
print(f'Mean words: {sub["wc"].mean():.1f}')
print(sub.head(10))
print(f'\nWritten to {OUTPUT}')